In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from itertools import chain

CATALOG = "vcredit_medallion"
SILVER_SCHEMA = f"{CATALOG}.silver"
GOLD_SCHEMA = f"{CATALOG}.gold"

In [0]:
def adicionar_sk(df, nome_sk, chave_natural):
    """Cria surrogate key sequencial ordenada pela chave natural, como primeira coluna."""
    janela = Window.orderBy(chave_natural)
    return df.withColumn(nome_sk, F.row_number().over(janela).cast("bigint")).select(nome_sk, *df.columns)


def gravar(df, nome):
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD_SCHEMA}.{nome}")
    )
    print(nome, spark.table(f"{GOLD_SCHEMA}.{nome}").count())

In [0]:
dim_motivo = adicionar_sk(
    spark.table(f"{SILVER_SCHEMA}.tb_base_motivos").select(
        "id_motivo", F.col("nome_motivo").alias("motivo"), "categoria", "criticidade"
    ),
    "sk_motivo", "id_motivo",
)

dim_canal = adicionar_sk(
    spark.table(f"{SILVER_SCHEMA}.tb_canais").select(
        "id_canal", F.col("nome_canal").alias("canal"), "tipo_canal"
    ),
    "sk_canal", "id_canal",
)

gravar(dim_motivo, "dim_motivo")
gravar(dim_canal, "dim_canal")

In [0]:
dim_atendente = adicionar_sk(
    spark.table(f"{SILVER_SCHEMA}.tb_base_atendentes").select(
        "id_atendente", "nome_atendente", "nivel_atendimento",
        "data_admissao", "data_desligamento", "status_atendente",
    ),
    "sk_atendente", "id_atendente",
)

# Membro desconhecido: chamados com atendente inválido (0, 99, 999) apontam pra cá em vez de ficar nulos
nao_identificado = spark.createDataFrame(
    [(-1, "Não identificado")], "sk_atendente bigint, nome_atendente string"
)
dim_atendente = dim_atendente.unionByName(nao_identificado, allowMissingColumns=True)

# A dim_cliente é a única tabela da gold que guarda o CPF (id_cliente); a fato usa só a sk_cliente
dim_cliente = adicionar_sk(
    spark.table(f"{SILVER_SCHEMA}.tb_clientes").select(
        "id_cliente", "regiao", "idade", "data_cadastro", "dia_vencimento_fatura"
    ),
    "sk_cliente", "id_cliente",
)

gravar(dim_atendente, "dim_atendente")
gravar(dim_cliente, "dim_cliente")

In [0]:
DATA_INICIO = "2025-01-01"
DATA_FIM = "2026-07-31"

nomes_mes = {1: "janeiro", 2: "fevereiro", 3: "março", 4: "abril", 5: "maio", 6: "junho",
             7: "julho", 8: "agosto", 9: "setembro", 10: "outubro", 11: "novembro", 12: "dezembro"}
nomes_dia = {1: "domingo", 2: "segunda", 3: "terça", 4: "quarta", 5: "quinta", 6: "sexta", 7: "sábado"}

# Transforma os dicionários em mapas do Spark para consultar pelo número
mapa_mes = F.create_map([F.lit(x) for x in chain(*nomes_mes.items())])
mapa_dia = F.create_map([F.lit(x) for x in chain(*nomes_dia.items())])

dim_data = (
    spark.sql(f"""
        SELECT explode(sequence(to_date('{DATA_INICIO}'), to_date('{DATA_FIM}'), interval 1 day)) AS data
    """)
    .withColumn("sk_data", F.date_format("data", "yyyyMMdd").cast("int"))  # chave no formato 20250101
    .withColumn("ano", F.year("data"))
    .withColumn("mes", F.month("data"))
    .withColumn("nome_mes", mapa_mes[F.month("data")])
    .withColumn("ano_mes", F.date_format("data", "yyyy-MM"))
    .withColumn("trimestre", F.quarter("data"))
    .withColumn("dia_semana", F.dayofweek("data"))  # 1 = domingo, 7 = sábado
    .withColumn("nome_dia_semana", mapa_dia[F.dayofweek("data")])
    .withColumn("fim_de_semana", F.dayofweek("data").isin(1, 7))
)
dim_data = dim_data.select("sk_data", *[c for c in dim_data.columns if c != "sk_data"])

gravar(dim_data, "dim_data")

In [0]:
custos_por_chamado = (
    spark.table(f"{SILVER_SCHEMA}.tb_custos")
    .groupBy("id_chamado")
    .agg(
        F.sum("custo").alias("custo_total"),
        F.count("*").alias("qtd_lancamentos_custo"),
        F.sum(F.col("custo_negativo").cast("int")).alias("qtd_custos_negativos"),
    )
)

pesquisas_por_chamado = (
    spark.table(f"{SILVER_SCHEMA}.tb_pesquisa_satisfacao")
    .groupBy("id_chamado")
    .agg(
        # when sem otherwise devolve nulo pras notas inválidas, e o avg ignora nulos
        F.avg(F.when(F.col("nota_valida"), F.col("nota_atendimento"))).alias("nota_media"),
        F.count("*").alias("qtd_pesquisas"),
    )
)

# Conferências: a soma não pode mudar e cada chamado tem que aparecer uma vez só
soma_silver = spark.table(f"{SILVER_SCHEMA}.tb_custos").agg(F.sum("custo")).first()[0]
soma_agregada = custos_por_chamado.agg(F.sum("custo_total")).first()[0]
print("soma custos silver:", soma_silver, "| agregada:", soma_agregada)
assert soma_silver == soma_agregada, "Soma de custos mudou na agregação"

for nome, df in [("custos", custos_por_chamado), ("pesquisas", pesquisas_por_chamado)]:
    linhas = df.count()
    ids = df.select("id_chamado").distinct().count()
    nulos = df.filter(F.col("id_chamado").isNull()).count()
    print(f"{nome}: {linhas} chamados | ids distintos: {ids} | id nulo: {nulos}")
    assert linhas == ids and nulos == 0, f"{nome}: id_chamado repetido ou nulo após agregação"

In [0]:
chamados = spark.table(f"{SILVER_SCHEMA}.tb_chamados").select(
    "id_chamado",
    F.col("id_cliente").alias("id_cliente_chamados"),
    "id_atendente",
    "id_motivo",
    "id_canal",
    "resolvido",
)

hora = spark.table(f"{SILVER_SCHEMA}.tb_chamados_hora").select(
    "id_chamado",
    # Texto vazio ou "null" vira nulo de verdade
    F.when(~F.lower(F.trim("id_cliente")).isin("", "null"), F.trim("id_cliente")).alias("id_cliente_hora"),
    "hora_abertura_chamado",
    "abertura_no_periodo",
    "ordem_temporal_invalida",
    "tempo_espera_segundos",
    "tempo_atendimento_segundos",
    "tempo_total_segundos",
)

# Todos os joins são left a partir de chamados e todas as chaves da direita são únicas,
# então nenhum chamado some nem se repete aqui
base = (
    chamados
    .join(hora, "id_chamado", "left")
    .join(custos_por_chamado, "id_chamado", "left")
    .join(pesquisas_por_chamado, "id_chamado", "left")
)

# Fato ainda com chaves naturais; as surrogate keys entram na próxima etapa
fato_natural = base.filter(F.col("abertura_no_periodo") == True).select(
    "id_chamado",
    F.coalesce("id_cliente_hora", "id_cliente_chamados").alias("id_cliente"),
    "id_atendente",
    "id_motivo",
    "id_canal",
    F.to_date("hora_abertura_chamado").alias("data_abertura"),
    F.hour("hora_abertura_chamado").alias("hora_do_dia"),
    "hora_abertura_chamado",
    "resolvido",
    "tempo_espera_segundos",
    "tempo_atendimento_segundos",
    "tempo_total_segundos",
    "ordem_temporal_invalida",
    # Sem lançamento de custo fica nulo, que é diferente de custo zero
    "custo_total",
    F.coalesce("qtd_lancamentos_custo", F.lit(0)).alias("qtd_lancamentos_custo"),
    F.coalesce("qtd_custos_negativos", F.lit(0)).alias("qtd_custos_negativos"),
    "nota_media",
    F.coalesce("qtd_pesquisas", F.lit(0)).alias("qtd_pesquisas"),
)

# Troca as chaves naturais pelas surrogate keys das dimensões
d_cli = spark.table(f"{GOLD_SCHEMA}.dim_cliente").select("id_cliente", "sk_cliente")
d_ate = spark.table(f"{GOLD_SCHEMA}.dim_atendente").filter(F.col("sk_atendente") != -1).select("id_atendente", "sk_atendente")
d_mot = spark.table(f"{GOLD_SCHEMA}.dim_motivo").select("id_motivo", "sk_motivo")
d_can = spark.table(f"{GOLD_SCHEMA}.dim_canal").select("id_canal", "sk_canal")

fato_chamados = (
    fato_natural
    .join(d_cli, "id_cliente", "left")
    .join(d_ate, "id_atendente", "left")
    .join(d_mot, "id_motivo", "left")
    .join(d_can, "id_canal", "left")
    .withColumn("sk_atendente", F.coalesce("sk_atendente", F.lit(-1).cast("bigint")))
    .withColumn("sk_data", F.date_format("data_abertura", "yyyyMMdd").cast("int"))
    .select(
        "id_chamado",
        "sk_cliente", "sk_atendente", "sk_motivo", "sk_canal", "sk_data",
        "hora_do_dia", "hora_abertura_chamado",
        "resolvido",
        "tempo_espera_segundos", "tempo_atendimento_segundos", "tempo_total_segundos",
        "ordem_temporal_invalida",
        "custo_total", "qtd_lancamentos_custo", "qtd_custos_negativos",
        "nota_media", "qtd_pesquisas",
    )
)

# Conferências antes de gravar
total_chamados = chamados.count()
total_base = base.count()
sem_horario = base.filter(F.col("hora_abertura_chamado").isNull()).count()
fora_periodo = base.filter(F.col("hora_abertura_chamado").isNotNull() & ~F.col("abertura_no_periodo")).count()
total_natural = fato_natural.count()
total_fato = fato_chamados.count()
ids_fato = fato_chamados.select("id_chamado").distinct().count()

print(f"chamados silver: {total_chamados} | após joins: {total_base}")
print(f"sem horário: {sem_horario} | fora do período: {fora_periodo} | fato: {total_fato}")
print(f"ids distintos na fato: {ids_fato}")

assert total_base == total_chamados, "Algum join duplicou ou perdeu chamados"
assert total_fato == total_natural, "A troca por surrogate keys duplicou ou perdeu chamados"
assert total_fato == ids_fato, "id_chamado repetido na fato"
assert total_fato > 0, "Fato vazia"
for sk in ["sk_cliente", "sk_motivo", "sk_canal", "sk_data"]:
    assert fato_chamados.filter(F.col(sk).isNull()).count() == 0, f"{sk} nula na fato"

gravar(fato_chamados, "fato_chamados")

In [0]:
# Integridade referencial: toda chave da fato tem que existir na dimensão
fato = spark.table(f"{GOLD_SCHEMA}.fato_chamados")

checagens = [
    ("sk_cliente", "dim_cliente"),
    ("sk_atendente", "dim_atendente"),
    ("sk_motivo", "dim_motivo"),
    ("sk_canal", "dim_canal"),
    ("sk_data", "dim_data"),
]

for chave, dim in checagens:
    dim_df = spark.table(f"{GOLD_SCHEMA}.{dim}").select(chave)
    nulos = fato.filter(F.col(chave).isNull()).count()
    sem_par = fato.filter(F.col(chave).isNotNull()).join(dim_df, chave, "left_anti").count()
    print(f"{chave}: nulo {nulos} | sem par na {dim}: {sem_par}")

print("chamados com atendente não identificado:", fato.filter(F.col("sk_atendente") == -1).count())